# DDPG

In [ ]:
# Minimal DDPG (continuous) — PyTorch + Gymnasium
# ----------------------------------------------------------
# - Deterministic actor + critic with target networks
# - Replay buffer, Polyak target updates
# - Ornstein-Uhlenbeck exploration noise
# ----------------------------------------------------------

import random
from collections import deque, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

# ----------------------------
# Hyperparameters
# ----------------------------
ENV_ID          = "Pendulum-v1"
TOTAL_STEPS     = 200_000
WARMUP_STEPS    = 1_000          # random exploration before learning
UPDATE_EVERY    = 1
BATCH_SIZE      = 128
BUFFER_CAPACITY = 100_000
GAMMA           = 0.99
TAU             = 0.005
LR_ACTOR        = 1e-3
LR_CRITIC       = 1e-3
SEED            = 42
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Replay buffer
# ----------------------------
Transition = namedtuple("Transition", ("state", "action", "next_state", "reward", "done"))

class ReplayBuffer:
    def __init__(self, capacity=BUFFER_CAPACITY):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        return Transition(*zip(*batch))
    def __len__(self): return len(self.buf)

# ----------------------------
# Networks
# ----------------------------
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, act_low, act_high, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim), nn.Tanh()
        )
        self.act_scale = torch.as_tensor((act_high - act_low)/2.0, device=DEVICE)
        self.act_bias  = torch.as_tensor((act_high + act_low)/2.0, device=DEVICE)
    def forward(self, s):
        return self.net(s) * self.act_scale + self.act_bias

class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim + act_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        return self.net(x)

# ----------------------------
# Ornstein-Uhlenbeck noise
# ----------------------------
class OUNoise:
    def __init__(self, size, mu=0.0, theta=0.15, sigma=0.2):
        self.mu = mu; self.theta = theta; self.sigma = sigma
        self.size = size
        self.state = np.ones(self.size) * self.mu
    def reset(self): self.state = np.ones(self.size) * self.mu
    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return self.state

# ----------------------------
# DDPG Agent
# ----------------------------
class DDPGAgent:
    def __init__(self, env):
        self.env = env
        obs_dim = env.observation_space.shape[0]
        act_dim = env.action_space.shape[0]
        act_low, act_high = env.action_space.low, env.action_space.high

        self.actor = Actor(obs_dim, act_dim, act_low, act_high).to(DEVICE)
        self.actor_tgt = Actor(obs_dim, act_dim, act_low, act_high).to(DEVICE)
        self.actor_tgt.load_state_dict(self.actor.state_dict())

        self.critic = Critic(obs_dim, act_dim).to(DEVICE)
        self.critic_tgt = Critic(obs_dim, act_dim).to(DEVICE)
        self.critic_tgt.load_state_dict(self.critic.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=LR_ACTOR)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=LR_CRITIC)

        self.replay = ReplayBuffer()
        self.noise = OUNoise(act_dim)

    def act(self, s, noise=True):
        s_t = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            a = self.actor(s_t).cpu().numpy()[0]
        if noise:
            a += self.noise.sample()
        return np.clip(a, self.env.action_space.low, self.env.action_space.high)

    def update(self):
        if len(self.replay) < BATCH_SIZE:
            return

        batch = self.replay.sample(BATCH_SIZE)
        s = torch.as_tensor(np.stack(batch.state), dtype=torch.float32, device=DEVICE)
        a = torch.as_tensor(np.stack(batch.action), dtype=torch.float32, device=DEVICE)
        r = torch.as_tensor(np.stack(batch.reward), dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        d = torch.as_tensor(np.stack(batch.done), dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        ns= torch.as_tensor(np.stack(batch.next_state), dtype=torch.float32, device=DEVICE)

        # Critic update
        with torch.no_grad():
            a_tgt = self.actor_tgt(ns)
            q_tgt = self.critic_tgt(ns, a_tgt)
            y = r + GAMMA * (1.0 - d) * q_tgt
        q = self.critic(s, a)
        critic_loss = nn.functional.mse_loss(q, y)

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        self.critic_opt.step()

        # Actor update (maximize Q)
        actor_loss = -self.critic(s, self.actor(s)).mean()
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        self.actor_opt.step()

        # Polyak averaging
        with torch.no_grad():
            for p, p_tgt in zip(self.critic.parameters(), self.critic_tgt.parameters()):
                p_tgt.data.mul_(1 - TAU).add_(TAU * p.data)
            for p, p_tgt in zip(self.actor.parameters(), self.actor_tgt.parameters()):
                p_tgt.data.mul_(1 - TAU).add_(TAU * p.data)

# ----------------------------
# Training loop
# ----------------------------
def train_ddpg():
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=SEED)
    agent = DDPGAgent(env)

    total_steps, ep_ret, ep_len = 0, 0.0, 0
    returns_log = []

    while total_steps < TOTAL_STEPS:
        if total_steps < WARMUP_STEPS:
            action = env.action_space.sample()
        else:
            action = agent.act(obs)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.replay.push(obs, action, next_obs, reward, float(done))
        ep_ret += reward; ep_len += 1; total_steps += 1
        obs = next_obs

        # Update
        if total_steps >= WARMUP_STEPS and total_steps % UPDATE_EVERY == 0:
            agent.update()

        # Episode done
        if done:
            returns_log.append(ep_ret)
            obs, _ = env.reset()
            agent.noise.reset()
            ep_ret, ep_len = 0.0, 0

        if len(returns_log) and len(returns_log) % 10 == 0:
            print(f"Step {total_steps:7d} | AvgReturn(10) = {np.mean(returns_log[-10:]):.1f}")

    env.close()
    print("Training finished. AvgReturn(100):", np.mean(returns_log[-100:]) if len(returns_log) >= 100 else np.mean(returns_log))
    return agent

# ----------------------------
# Evaluation
# ----------------------------
def eval_policy(agent, episodes=5):
    env = agent.env
    avg_ret = 0.0
    with torch.no_grad():
        for _ in range(episodes):
            obs, _ = env.reset()
            done, ep_ret = False, 0.0
            while not done:
                s_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                a = agent.actor(s_t).cpu().numpy()[0]
                obs, r, terminated, truncated, _ = env.step(a)
                done = terminated or truncated
                ep_ret += r
            avg_ret += ep_ret
    return avg_ret / episodes

if __name__ == "__main__":
    agent = train_ddpg()
    avg = eval_policy(agent, episodes=10)
    print(f"Eval average return over 10 episodes: {avg:.2f}")
